In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# rerun strategy balancing analyses, epochs
# unit and sanity checks!!!
# add movement (average normalized me on a trial)
"""--------------------------------------------"""
# add time
# one regressor

## lite

In [ ]:
# no outlier trial filtering at the moment
# for session subsampling, i divided the number of tents so that the frequency of slow drift is the same

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id)
encoder.fit_encoder()
encoder.encoder_predict()
# encoder.fit_baseline()
# encoder.baseline_predict()

In [ ]:
encoder.verify(subtract_baseline=False)

In [ ]:
from squiggs.neuron_viewer import NeuronViewer
from squiggs.renderers import FitRenderer

r = FitRenderer(
    y=encoder.robs,
    yhat=encoder.robs_predict["encoder"],
    mode="lite",
)

nv = NeuronViewer(num_units=encoder.num_units, render_func=r)

In [ ]:
encoder.encoder.coef_.shape

In [ ]:
import numpy as np

np.where(encoder.reg_keys == 1)[0].shape

In [ ]:
encoder.reg_idxs["DLS"].shape

In [ ]:
encoder.psths["DMS"].shape

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts, get_tavg_sc_cond

"""
drift: 21, 29, 35, 36
response: 16, 25, 26, 28
"""

reg = "DMS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder.robs[:, encoder.reg_idxs[reg]], encoder.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder.dm_names,
    robs=encoder.robs[:, encoder.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder.trial_data, mode=mode),
    spike_times=encoder.spike_times[reg],
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=len(encoder.psths[reg]), render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(subj_id, sess_id)
se.plot_cvr2()
se.plot_dr2()

# session aggregates

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids, get_tavg_sc_cond

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]

scores_lite = {}
scores_lite_baseline = {}
scores_lite_ps_baseline = {}
coefs = {}
sc_tavgs = {"response": {}, "rewarded": {}}

for sess_id in sess_ids:
    encoder = Encoder(subj_id, sess_id, separate_drift=False)
    encoder.fit_encoder()
    encoder.get_r2()

    scores_lite[sess_id] = encoder.scores["encoder"]
    scores_lite_baseline[sess_id] = encoder.scores["baseline"]
    scores_lite_ps_baseline[sess_id] = encoder.scores["ps_baseline"]

    coefs[sess_id] = encoder.encoder.coef_

    sc_tavgs["response"][sess_id] = get_tavg_sc_cond(
        encoder.robs, encoder.trial_data, cond="response"
    )
    sc_tavgs["rewarded"][sess_id] = get_tavg_sc_cond(
        encoder.robs, encoder.trial_data, cond="rewarded"
    )

In [ ]:
scores_lite_ps_baseline_ = np.concatenate(
    [scores_lite_ps_baseline[sess_id] for sess_id in sess_ids]
)

In [ ]:
scores_lite_ = np.concatenate([scores_lite[sess_id] for sess_id in sess_ids])
scores_lite_baseline_ = np.concatenate(
    [scores_lite_baseline[sess_id] for sess_id in sess_ids]
)
coefs_ = np.concatenate([coefs[sess_id] for sess_id in sess_ids])

In [ ]:
scores_lite_sd = {}
scores_lite_baseline_sd = {}

for sess_id in sess_ids:
    encoder = Encoder(subj_id, sess_id, separate_drift=True)
    encoder.fit_encoder()
    encoder.get_r2()

    scores_lite_sd[sess_id] = encoder.scores["encoder"]
    scores_lite_baseline_sd[sess_id] = encoder.scores["baseline"]

scores_lite_sd_ = np.concatenate([scores_lite_sd[sess_id] for sess_id in sess_ids])
scores_lite_baseline_sd_ = np.concatenate(
    [scores_lite_baseline_sd[sess_id] for sess_id in sess_ids]
)

In [ ]:
np.where((scores_lite_sd_ > scores_lite_baseline_sd_) & (scores_lite_sd_ > 0))[0].shape[
    0
] / len(scores_lite_sd_)

In [ ]:
np.where((scores_lite_sd_ > scores_lite_baseline_sd_) & (scores_lite_sd_ > 0))[0].shape

In [ ]:
np.where(scores_lite_sd_ > scores_lite_baseline_sd_)[0].shape

In [ ]:
np.where(scores_lite_sd_ > 0)[0].shape

In [ ]:
np.where((scores_lite_ > scores_lite_baseline_) & (scores_lite_ > 0))[0].shape[0] / len(
    scores_lite_
)

In [ ]:
np.where((scores_lite_ > scores_lite_baseline_) & (scores_lite_ > 0))[0].shape

In [ ]:
np.where((scores_lite_ > scores_lite_baseline_))[0].shape

In [ ]:
np.where(scores_lite_ > 0)[0].shape

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(scores_lite_sd_.reshape(-1, 1), scores_lite_)

plt.figure(figsize=(3, 3), tight_layout=True)
plt.scatter(scores_lite_sd_, scores_lite_, alpha=0.5, s=0.5)

plt.plot([-0.2, 1], [-0.2, 1], color="#666666", label="unity")
plt.plot(
    [-0.2, 1],
    [lr.coef_[0] * -0.2 + lr.intercept_, lr.coef_[0] + lr.intercept_],
    color="#BB3333",
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.xlabel(r"$r^2$, separate drift")
plt.ylabel(r"$r^2$, fit together")
plt.title(
    f"tgt > sd (encoder): {np.where(scores_lite_ > scores_lite_sd_)[0].shape[0]}/{len(scores_lite_)}={np.where(scores_lite_ > scores_lite_sd_)[0].shape[0] / len(scores_lite_):.3f}"
)
plt.legend()
plt.show()

In [ ]:
scores_lite_ps_baseline_

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(
    scores_lite_baseline_sd_.reshape(-1, 1), scores_lite_ps_baseline_
)

plt.figure(figsize=(3, 3), tight_layout=True)
plt.scatter(scores_lite_baseline_sd_, scores_lite_ps_baseline_, alpha=0.5, s=0.5)

plt.plot([-0.2, 1], [-0.2, 1], color="#666666", label="unity")
plt.plot(
    [-0.2, 1],
    [lr.coef_[0] * -0.2 + lr.intercept_, lr.coef_[0] + lr.intercept_],
    color="#BB3333",
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.xlabel(r"$r^2$, separate drift")
plt.ylabel(r"$r^2$, pseudo baseline, fit together")
plt.title(
    f"ps_baseline > sd_baseline: {np.where(scores_lite_ps_baseline_ > scores_lite_baseline_sd_)[0].shape[0]}/{len(scores_lite_)}={np.where(scores_lite_ps_baseline_ > scores_lite_baseline_sd_)[0].shape[0] / len(scores_lite_):.3f}"
)
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(
    scores_lite_baseline_sd_.reshape(-1, 1), scores_lite_baseline_
)

plt.figure(figsize=(3, 3), tight_layout=True)
plt.scatter(scores_lite_baseline_sd_, scores_lite_baseline_, alpha=0.5, s=0.5)

plt.plot([-0.2, 1], [-0.2, 1], color="#666666", label="unity")
plt.plot(
    [-0.2, 1],
    [lr.coef_[0] * -0.2 + lr.intercept_, lr.coef_[0] + lr.intercept_],
    color="#BB3333",
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.xlabel(r"$r^2$, separate drift")
plt.ylabel(r"$r^2$, fit drift together")
plt.title(
    f"lite > lite_sd: {np.where(scores_lite_baseline_ > scores_lite_baseline_sd_)[0].shape[0]}/{len(scores_lite_)}={np.where(scores_lite_ > scores_lite_sd_)[0].shape[0] / len(scores_lite_):.3f}"
)
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(scores_lite_sd_.reshape(-1, 1), scores_lite_)

plt.figure(figsize=(3, 3), tight_layout=True)
plt.scatter(scores_lite_sd_, scores_lite_, alpha=0.5, s=0.5)

plt.plot([-0.2, 1], [-0.2, 1], color="#666666", label="unity")
plt.plot(
    [-0.2, 1],
    [lr.coef_[0] * -0.2 + lr.intercept_, lr.coef_[0] + lr.intercept_],
    color="#BB3333",
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.xlabel(r"$r^2$, separate drift")
plt.ylabel(r"$r^2$, fit drift together")
plt.title(
    f"lite > lite_sd: {np.where(scores_lite_ > scores_lite_sd_)[0].shape[0]}/{len(scores_lite_)}={np.where(scores_lite_ > scores_lite_sd_)[0].shape[0] / len(scores_lite_):.3f}"
)
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.hist(scores_lite_, bins=np.linspace(-0.2, 1, 12), alpha=0.5)
plt.hist(scores_lite_sd_, bins=np.linspace(-0.2, 1, 12), alpha=0.5)
plt.show()

In [ ]:
sc_tavgs_ = {
    cond: {
        key: np.concatenate([sc_tavgs[cond][sess_id][key] for sess_id in sess_ids])
        for key in sc_tavgs[cond][sess_ids[0]].keys()
    }
    for cond in sc_tavgs.keys()
}

In [ ]:
cond = "rewarded"

if cond == "response":
    keys = ["left", "right"]
    idxs = [6, 5]
elif cond == "rewarded":
    keys = ["corr", "incorr"]
    idxs = [8, 7]

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(4, 2), tight_layout=True)

for i in range(2):
    axes[i].scatter(sc_tavgs_[cond][keys[i]], coefs_[:, idxs[i]], s=0.5, alpha=0.5)
    axes[i].axhline(y=0, color="k", linewidth=0.5)
    axes[i].axvline(x=0, color="k", linewidth=0.5)

    axes[i].set_xlabel(f"avg norm sc, {keys[i]}")
    axes[i].set_ylabel(f"beta weight, {keys[i]}")

In [ ]:
fig, ax = plt.subplots(tight_layout=True)

ax.scatter(scores_lite_baseline_, scores_lite_, s=0.5, alpha=0.5)
ax.plot([-0.5, 1], [-0.5, 1], color="#666666", linestyle="--", linewidth=0.5)
ax.axhline(y=0, color="k", linewidth=0.5)
ax.axvline(x=0, color="k", linewidth=0.5)

ax.set_xlabel(r"$r^2$, baseline")
ax.set_ylabel(r"$r^2$, encoder")

## cvr2 and delta r2

In [ ]:
from sg.models import ShuffledEncoder

cvr2s = {}
dr2s = {}

for sess_id in sess_ids:
    print(sess_id)
    se = ShuffledEncoder(subj_id, sess_id)
    se.get_cvr2_all()
    se.get_dr2_all()

    cvr2s[sess_id] = se.cvr2
    dr2s[sess_id] = se.dr2

In [ ]:
cvr2s_ = {
    pivot: np.array([cvr2s[sess_id][pivot] for sess_id in sess_ids])
    for pivot in encoder.task_vars
}
dr2s_ = {
    pivot: np.array([dr2s[sess_id][pivot] for sess_id in sess_ids])
    for pivot in encoder.task_vars
}

In [ ]:
cvr2_mean = [cvr2s_[pivot].mean() for pivot in encoder.task_vars]
cvr2_std = [cvr2s_[pivot].std() for pivot in encoder.task_vars]

fig, ax = plt.subplots(tight_layout=True)
ax.bar(encoder.task_vars, cvr2_mean, width=0.5)
ax.errorbar(
    x=encoder.task_vars,
    y=cvr2_mean,
    yerr=cvr2_std,
    color="k",
    fmt=".",
    capsize=2,
)
ax.set_ylabel(r"cv $r^2$")
ax.tick_params(axis="x", labelrotation=45)

In [ ]:
dr2_mean = [dr2s_[pivot].mean() for pivot in encoder.task_vars]
dr2_std = [dr2s_[pivot].std() for pivot in encoder.task_vars]

fig, ax = plt.subplots(tight_layout=True)
ax.bar(encoder.task_vars, dr2_mean, width=0.5)
ax.errorbar(
    x=encoder.task_vars,
    y=dr2_mean,
    yerr=dr2_std,
    color="k",
    fmt=".",
    capsize=2,
)
ax.set_ylabel(r"$\Delta r^2$")
ax.tick_params(axis="x", labelrotation=45)

In [ ]:
# sanity check passed
se.encoder_full.verify()
# se.encoder_shuffle.verify()

## v. liska

In [ ]:
from core.data import subject_ids, session_ids

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]

In [ ]:
import numpy as np
from sg.fitter import LVMFamily

n_folds = 5

scores_liska = {}
scores_liska_baseline = {}

for sess_id in sess_ids:
    for i in range(n_folds):
        family = LVMFamily(
            subj_id=subj_id,
            sess_id=sess_id,
            n_latents_mult=1,
            n_latents_addt=1,
            sanity_check=0,
            regions=None,
            refit=False,
            balance_strategy=False,
            task_vars=[
                "response",
                "rewarded",
                "block_side",
                "response_prev",
                "rewarded_prev",
            ],
            n_splines=5,
            tpre=0.5,
            tpost=1,
            binwidth_ms=25,
            alignment="choice",
            tv_reg={"l2": 0.1},
            seed=i,
        )

        family.fit_all(fit_lvms=False, update_cids=False)
        family.eval()

        if i == 0:
            num_units = family.num_units
            scores_liska_baseline_cv = np.zeros((n_folds, num_units))
            scores_liska_cv = np.zeros((n_folds, num_units))

        scores_liska_baseline_cv[i] = family.res_baseline["r2test"]
        scores_liska_cv[i] = family.res_taskvar["r2test"]

    scores_liska_baseline[sess_id] = np.median(scores_liska_baseline_cv, axis=0)
    scores_liska[sess_id] = np.median(scores_liska_cv, axis=0)

In [ ]:
scores_liska_ = np.concatenate([scores_liska[sess_id] for sess_id in sess_ids])
scores_liska_baseline_ = np.concatenate(
    [scores_liska_baseline[sess_id] for sess_id in sess_ids]
)

In [ ]:
plt.figure(tight_layout=True)
plt.scatter(scores_liska_, scores_lite_, s=0.5, alpha=0.5)
plt.plot([-1.5, 1], [-1.5, 1], color="#666666", linestyle="--", linewidth=0.5)
plt.axhline(y=0, linewidth=0.5, color="k")
plt.axvline(x=0, linewidth=0.5, color="k")

plt.xlabel("$r^2$, liska")
plt.ylabel("$r^2$, lite")
plt.show()

In [ ]:
scores_liska_baseline_.shape

In [ ]:
n_units = len(scores_lite_)
n_cids = np.where((scores_lite_ > 0) & (scores_lite_ > scores_lite_baseline_))[0].shape[
    0
]

plt.figure()
plt.pie(
    [n_units - n_cids, n_cids],
    colors=[
        "#666666",
        "#F1AEAE",
    ],
    autopct="%.1f%%",
    startangle=90,
)
plt.show()

n_cids_liska = np.where((scores_liska_ > 0) & (scores_liska_ > scores_liska_baseline_))[
    0
].shape[0]

plt.figure()
plt.pie(
    [n_units - n_cids_liska, n_cids_liska],
    colors=[
        "#666666",
        "#F1AEAE",
    ],
    autopct="%.1f%%",
    startangle=90,
)
plt.show()